In [1]:
import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from typing import Any

from project_root import PROJECT_ROOT

import fiftyone as fo
import torchvision

In [2]:
coco_root = Path("/home/dherrera/data/coco")
data_root = Path("/home/dherrera/data/elephants")

In [3]:
from scripts.model_serialization import load_model


# model_base = load_model(
#     "/home/dherrera/Downloads/train_output/model_base.pth", weights_only=True
# )
# model_trained = load_model(
#     "/home/dherrera/git/zoo_vision/models/model_coco_4cam_8k/maskrcnn_c2_coco_4cam_8k.pth",
#     weights_only=True,
# )
model_trained = load_model(
    PROJECT_ROOT / "models/identity_1layer/dense121_c5_identity.ptc"
)
model_trained

RecursiveScriptModule(
  original_name=ModelWithTransforms
  (model): RecursiveScriptModule(
    original_name=DenseNet
    (features): RecursiveScriptModule(
      original_name=Sequential
      (conv0): RecursiveScriptModule(original_name=Conv2d)
      (norm0): RecursiveScriptModule(original_name=BatchNorm2d)
      (relu0): RecursiveScriptModule(original_name=ReLU)
      (pool0): RecursiveScriptModule(original_name=MaxPool2d)
      (denseblock1): RecursiveScriptModule(
        original_name=_DenseBlock
        (denselayer1): RecursiveScriptModule(
          original_name=_DenseLayer
          (norm1): RecursiveScriptModule(original_name=BatchNorm2d)
          (relu1): RecursiveScriptModule(original_name=ReLU)
          (conv1): RecursiveScriptModule(original_name=Conv2d)
          (norm2): RecursiveScriptModule(original_name=BatchNorm2d)
          (relu2): RecursiveScriptModule(original_name=ReLU)
          (conv2): RecursiveScriptModule(original_name=Conv2d)
        )
        (dense

In [4]:
with (PROJECT_ROOT / "data/config.json").open() as f:
    config = json.load(f)
classes = [(data["id"], name) for name, data in config["individuals"].items()]
classes = sorted(classes, key=lambda x: x[0])
classes = [f"{x[0]:02}_{x[1]}" for x in classes]
print(classes)

['01_Chandra', '02_Indi', '03_Fahra', '04_Panang', '05_Thai']


In [5]:
# from PIL import Image
# for sample in ds_zoo_elephants.iter_samples():
#   im = np.asarray(Image.open(sample.filepath).convert("RGB"))
#   result = fo_model.predict(im)
#   print(result)
#   sample.add_labels(result, label_field="zoo_identity")
#   break

In [6]:
import fiftyone.utils.torch as fout

ds_zoo_elephants = fo.load_dataset("zoo-elephants-identity")
session = fo.launch_app(ds_zoo_elephants, auto=False)

# class MyOutputProcessor:
#   def __init__(self):
#     pass
#   def
config = fout.TorchImageModelConfig(
    {
        "entrypoint_fcn": lambda: model_trained,
        "entrypoint_args": {},
        "output_processor": fout.ClassifierOutputProcessor(classes=classes),
        "classes": classes,
        # "transforms": torchvision.transforms.PILToTensor(),
        "image_min_dim": 224,
        "image_max_dim": 2048,
    }
)
fo_model = fout.TorchImageModel(config)

predictions_view = ds_zoo_elephants.take(1000, seed=51)
predictions_view.apply_model(fo_model, label_field="zoo_identity")
# high_conf_view = predictions_view.filter_labels("zoo maskrcnn", fo.ViewField("confidence") > 0.85, only_matches=False)

session.view = predictions_view
session.open_tab()

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
Session launched. Run `session.show()` to open the App in a cell output.
 100% |███████████████| 1000/1000 [9.4s elapsed, 0s remaining, 90.4 samples/s]      


<IPython.core.display.Javascript object>